# 🤖 Simulator‐Based Multi‐Turn Testing (Single `simulation_config.yaml`)

This notebook runs a **RedTeamingOrchestrator** (multi‐turn simulation) against your HTTP assistant endpoint.
All parameters (endpoints, request templates, parsing rules, objectives, reporter settings, etc.) are loaded from a single **`simulation_config.yaml`**.  

**Advantages**:
- Product teams can edit exactly one file (`simulation_config.yaml`) if anything changes.
- The notebook itself does not require modification after initial setup.

**Overview**:  
1. Load environment variables and `simulation_config.yaml`  
2. Build a `MultiFieldResponseParser` + thread-ID helpers  
3. Instantiate `HTTPTargetX`, `Evaluator`, and `RedTeamingOrchestrator`
4. Loop over each objective in `simulation_config.yaml` → run `run_simulation_async`  
5. Generate a HTML report (via `generate_simulation_report`)  

> **Before you begin**:  
> - Ensure your `.env` contains:
>   ```ini
>   TARGET_ENDPOINT=<your_endpoint_prefix>
>   AUTH_TOKEN=<your_api_token>
>   ```  
> - Put this notebook and **`simulation_config.yaml`** in the same directory.  
> - Make sure `strategy_path`, `scorer_path`, and any other file paths in `simulation_config.yaml` are correct.


In [ ]:
# Cell 1: Imports, logging, and load both .env + config.yaml

import logging
import os
import re
import time
import asyncio
import yaml
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

# PyRIT (initialize in-memory DuckDB)
from pyrit.common import IN_MEMORY, initialize_pyrit
from pyrit.prompt_target import MultiFieldResponseParser, OpenAIChatTarget
from pyrit.prompt_target import HTTPTargetX
from pyrit.score.evaluator import Evaluator
from pyrit.orchestrator import RedTeamingOrchestrator
from pyrit.common.report_generator import get_conversation_report_async
from pyrit.common.report_generator import create_report

from typing import Optional

logging.basicConfig(level=logging.WARNING)

# Initialize PyRIT memory
initialize_pyrit(memory_db_type=IN_MEMORY)

# Load environment variables (so {token} and {endpoint/members} are available)
load_dotenv()

# Read config.yaml
config_path = Path("simulation.yaml")
with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

# Extract each section from config.yaml
dataset_cfg         = cfg  # (we only have one top‐level YAML; name kept for clarity)
http_request_raw    = dataset_cfg["http_request_raw"]
field_defs          = dataset_cfg["field_defs"]
thread_id_pattern   = dataset_cfg["thread_id_pattern"]
strategy_path       = dataset_cfg["strategy_path"]
scorer_path         = dataset_cfg["scorer_path"]
objectives          = dataset_cfg["objectives"]
max_turns           = dataset_cfg["max_turns"]
timeout_seconds     = dataset_cfg["timeout_seconds"]
use_score_as_feedback = dataset_cfg["use_score_as_feedback"]
scorer_type         = dataset_cfg["scorer_type"]
report_path         = dataset_cfg["report_path"]
thread_id_key       = dataset_cfg["thread_id_query_param_key"]

# Build “base_url” + “token” from environment (dot‐env):
base_url   = os.getenv("TARGET_ENDPOINT")
token      = os.getenv("AUTH_TOKEN")

# Substitute into the raw HTTP template.
# Because YAML uses double {{…}} around PROMPT, Python .format() will collapse to single {PROMPT}.
http_request_templated = http_request_raw.format(
    base_url=base_url,
    token=token
)


## 🔍 Build `MultiFieldResponseParser` & Thread-ID Helpers

- **`MultiFieldResponseParser`**:  
  Reads `field_defs` from `simulation_config.yaml`.  
- **`thread_id_parser`**:  
  Uses the regex in `thread_id_pattern` from `simulation_config.yaml` to extract new thread IDs.  
- **`thread_id_injector`**:  
  Strips any existing `threadId and appends it to the URL.


In [ ]:
# Cell 2: Parser + Thread‐ID helpers

import requests

# 2.1 MultiFieldResponseParser from YAML’s field_defs
multi_parser = MultiFieldResponseParser(field_definitions=field_defs)

# 2.2 thread_id_parser using the YAML‐provided regex
def thread_id_parser(response: requests.Response) -> Optional[str]:
    text = response.content.decode("utf-8")
    match = re.search(thread_id_pattern, text)
    return match.group(1) if match else None

# 2.3 thread_id_injector (appends threadId received to the next request)
def thread_id_injector(raw_http_request: str, thread_id: str, thread_id_key: str = "threadId") -> str:
    """
    Injects or replaces the `{thread_id_key}` query parameter in the first URL found in the raw HTTP request.
    """
    import re

    url_pattern = r"(https?://[^\s]+)"
    m = re.search(url_pattern, raw_http_request)
    if not m:
        raise ValueError("No URL found in raw HTTP request; cannot inject thread ID.")

    original_url = m.group(1)

    # Remove existing threadId (or other key) if present
    pattern = rf"([?&]){re.escape(thread_id_key)}=[^&]*"
    cleaned = re.sub(pattern, "", original_url)

    sep = "&" if "?" in cleaned else "?"
    new_url = f"{cleaned}{sep}{thread_id_key}={thread_id}"

    return raw_http_request.replace(original_url, new_url)


## 🚀 Instantiate `HTTPTargetX`, `Evaluator`, and `RedTeamingOrchestrator`

1. **`HTTPTargetX`**:  
   - `http_request` = `http_request_templated` (from Cell 2).  
   - `prompt_regex_string` = `"{PROMPT}"` (standard).  
   - `use_tls=True` by default.  
   - `response_parser=multi_parser` (Cell 4).  
   - `thread_id_parser=thread_id_parser` (Cell 4).  
   - `timeout=timeout_seconds` (from `simulation_config.yaml`).  

2. **`Evaluator`**:  
   Uses `OpenAIChatTarget()` plus `scorer_path` (from `simulation_config.yaml`), and `scorer_type`.  

3. **`RedTeamingOrchestrator`**:
   - `objective_target` = `http_prompt_target`  
   - `adversarial_chat` = `OpenAIChatTarget()` (to generate adversarial prompts)  
   - `adversarial_chat_system_prompt_path` = `strategy_path` (system prompt YAML)  
   - `objective_scorer` = `objective_scorer` (Evaluator instance)  
   - `use_score_as_feedback` = from `simulation_config.yaml`  
   - `evaluate_chat=True` (we want to re‐score entire chat each turn)  
   - `max_turns` = from `simulation_config.yaml`  
   - `thread_id_injector` = our helper (Cell 4)  
   - `scorer_type` = from `simulation_config.yaml`  
   - `verbose=True` (so you can see printouts)


In [ ]:
# Cell 3: Instantiate HTTPTargetX, Evaluator, and RedTeamingOrchestrator

# 6.1 HTTP target using our templated raw request, parser, and thread‐ID helpers:
http_prompt_target = HTTPTargetX(
    http_request        = http_request_templated,
    prompt_regex_string = "{PROMPT}",
    use_tls             = True,
    response_parser     = multi_parser,
    thread_id_parser    = thread_id_parser,
    timeout             = timeout_seconds
)

## 🏃‍♀️ Run Each Objective via `run_simulation_async`

Below, we define:
1. **`run_simulation()`**:  
   - Loops over each string in `objectives` (from `simulation_config.yaml`).  
   - For each:
     - Create a fresh `OpenAIChatTarget()` (adversarial chat).
     - Create a new `Evaluator(...)` with `additional_evaluator_variables={"objective": objective}`.
     - Instantiate a new `RedTeamingOrchestrator(...)` with all parameters from `simulation_config.yaml`.
     - Call `await orchestrator.run_simulation_async(objective=objective)`.
     - Collect its conversation report (`get_conversation_report_async()`).
   - Return a list of those reports.

2. **`generate_report()`**:  
   - Writes out a single HTML file under `report_path` (from `simulation_config.yaml`), named with a timestamp.
   - Calls `generate_simulation_report(...)` to produce the final HTML.


In [ ]:
from pyrit.prompt_normalizer import PromptNormalizer
from pyrit.attacks import MultiTurnAttackContext, RedTeamingAttack, AttackConverterConfig, AttackScoringConfig, AttackAdversarialConfig

async def run_simulation():
    """
    Loops over each objective in config.yaml → runs RedTeamingAttack → returns list of reports.
    """
    reports = []
    start_time = time.time()

    for objective in objectives:
        # 1) Per-objective scorer variables
        scorer_vars = {"restricted_topic": objective}

        # 2) Clone base Evaluator & inject per-objective variables
        evaluator_instance = Evaluator(
            chat_target                    = OpenAIChatTarget(),
            evaluator_yaml_path            = Path(scorer_path),
            additional_evaluator_variables = scorer_vars,
            scorer_type                    = scorer_type
        )

        # 3) Build RedTeamingAttack (not orchestrator!)
        attack_adversarial_config = AttackAdversarialConfig(
            target                      = OpenAIChatTarget(),
            system_prompt_path          = Path(strategy_path),
            seed_prompt                 = "",  # or your seed prompt string/SeedPrompt
        )
        attack_converter_config = AttackConverterConfig(
            request_converters=[],
            response_converters=[]
        )
        attack_scoring_config = AttackScoringConfig(
            objective_scorer            = evaluator_instance,
            auxiliary_scorers           = [],
            use_score_as_feedback       = use_score_as_feedback,
            successful_objective_threshold = 0.8  # adjust as needed
        )

        red_teaming_attack = RedTeamingAttack(
            objective_target            = http_prompt_target,
            attack_adversarial_config   = attack_adversarial_config,
            attack_converter_config     = attack_converter_config,
            attack_scoring_config       = attack_scoring_config,
            prompt_normalizer           = PromptNormalizer(),
            evaluate_chat               = True,
            scorer_type                 = scorer_type,
            thread_id_injector          = thread_id_injector,
            max_retries                 = 1,
            max_turns                   = max_turns,
        )

        # 4) Run the red-teaming simulation asynchronously
        context = MultiTurnAttackContext(
            objective=objective,
            memory_labels={},
        )

        sim_result = await red_teaming_attack.execute_with_context_async(context=context)

        # 5) Pull back its conversation report (dict) and append
        reports.append(await get_conversation_report_async(sim_result))

    elapsed = time.time() - start_time
    print(f"✅ All red-teaming simulations done in {elapsed:.2f} seconds")
    return reports

async def generate_report(results: list, execution_time: float):
    """
    Writes out a single HTML under report_path with a timestamp.
    """
    report_dir = Path(report_path).resolve()
    report_dir.mkdir(parents=True, exist_ok=True)

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"simulation_report_{ts}.html"

    create_report(
        results        = results,
        save_path      = report_dir / fname,
        description    = (
            "This report provides an overview of multi-turn customer simulations. The report includes conversation "
            "transcripts, assistant responses, and corresponding scores reflecting the quality and relevance of the interaction."
        ),
        execution_time = execution_time
    )

## 🎬 Kick‐Off: `main()`

1. Call `await run_simulation()` → get a list of “conversation reports” (one dict per objective).  
2. Measure elapsed time, then hand both to `generate_report(...)`.  
3. The final HTML appears under `report_path` (from `simulation_config.yaml`) with a timestamp.


In [ ]:
# Cell 5: Main async runner

async def main():
    start_time = time.time()
    results = await run_simulation()
    exec_time = time.time() - start_time
    await generate_report(results, exec_time)

# Execute
await main()
